# Phase 9 Audit and Analysis
This notebook executes the post-training analysis (Routing stats, Error Profiles, Qualitative Visualizations) for Phase 9.

In [ ]:
import os, sys, glob, torch, json
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import yaml

# Ensure we are in the repo
if not os.path.exists("vision_transformer_research"):
    !git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git vision_transformer_research
os.chdir("vision_transformer_research")
sys.path.append(os.path.abspath("."))

from datasets.medical_dataset import MedicalImageMaskDataset, get_medical_transforms
from torch.utils.data import DataLoader
from models.swin import SwinSegmentation, replace_swin_ffn_with_standard_moe
from evaluation.metrics import SegmentationMetricAccumulator

In [ ]:
# 1. LOAD DATASETS
_, transform = get_medical_transforms(224)
busi_dataset = MedicalImageMaskDataset("data/medical_dataset", transform=transform)
busbra_dataset = MedicalImageMaskDataset("data/busbra_dataset", transform=transform)

busi_loader = DataLoader(busi_dataset, batch_size=8, shuffle=False)
busbra_loader = DataLoader(busbra_dataset, batch_size=8, shuffle=False)

In [ ]:
# 2. LOAD MODELS
device = "cuda"

# Vanilla Swin
vanilla_paths = sorted(glob.glob("outputs/swin_vanilla_*/checkpoints/best_model.pth"))
model_vanilla = SwinSegmentation(num_classes=3, pretrained=False)
if vanilla_paths:
    ckpt = torch.load(vanilla_paths[-1], map_location=device)
    model_vanilla.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt, strict=False)
model_vanilla.to(device).eval()

# Standard-MLP MoE
moe_paths = sorted(glob.glob("outputs/swin_standard_moe_*/checkpoints/best_model.pth"))
with open("configs/swin_standard_moe_kaggle.yaml", "r") as f:
    cfg_moe = yaml.safe_load(f)
model_moe = SwinSegmentation(num_classes=3, pretrained=False)
replace_swin_ffn_with_standard_moe(model_moe, cfg_moe["moe"])
if moe_paths:
    ckpt = torch.load(moe_paths[-1], map_location=device)
    model_moe.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt, strict=False)
model_moe.to(device).eval()

print("Models loaded.")

In [ ]:
# 3. ERROR PROFILE ANALYSIS (Vanilla vs MoE on BUSBRA)
def evaluate_model(model, loader):
    acc = SegmentationMetricAccumulator(num_classes=2)
    with torch.no_grad():
        for imgs, masks in tqdm(loader):
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            preds = (logits.argmax(dim=1) > 0).long()
            acc.update(preds, masks)
    return acc.compute()

if vanilla_paths and moe_paths:
    print("Evaluating Vanilla Swin...")
    vanilla_metrics = evaluate_model(model_vanilla, busbra_loader)
    print("Evaluating Standard-MLP MoE...")
    moe_metrics = evaluate_model(model_moe, busbra_loader)
    
    print("\n=== ERROR PROFILE COMPARISON (BUSBRA) ===")
    print(f"Vanilla -> TP: {vanilla_metrics['class_tp'][1]}, FP: {vanilla_metrics['class_fp'][1]}, FN: {vanilla_metrics['class_fn'][1]}")
    print(f"MoE     -> TP: {moe_metrics['class_tp'][1]}, FP: {moe_metrics['class_fp'][1]}, FN: {moe_metrics['class_fn'][1]}")
    
    diff_fp = vanilla_metrics['class_fp'][1] - moe_metrics['class_fp'][1]
    diff_fn = vanilla_metrics['class_fn'][1] - moe_metrics['class_fn'][1]
    print(f"MoE Reduced False Positives by: {diff_fp}")
    print(f"MoE Reduced False Negatives by: {diff_fn}")
else:
    print("Cannot run error profile: Missing checkpoints!")

In [ ]:
# 4. ROUTING BEHAVIOR ANALYSIS
def reset_routing_stats(model):
    for module in model.modules():
        if hasattr(module, "routing_stats"):
            module.routing_stats = []

def collect_routing_stats(model, loader):
    reset_routing_stats(model)
    with torch.no_grad():
        for imgs, _ in tqdm(loader):
            model(imgs.to(device))
            
    global_gates = []
    for module in model.modules():
        if hasattr(module, "routing_stats") and len(module.routing_stats) > 0:
            # Concat all batches for this layer
            layer_gates = torch.cat([s["gates"] for s in module.routing_stats], dim=0)
            global_gates.append(layer_gates)
            
    if not global_gates: return [0,0,0,0], 0
    
    all_gates = torch.cat(global_gates, dim=0) # [Total_Tokens, Num_Experts]
    # Calculate utilization % (how often each expert was in top-k)
    utilization = all_gates.mean(dim=0).cpu().numpy()
    utilization = (utilization / utilization.sum()) * 100
    
    entropy = -torch.sum(all_gates.mean(dim=0) * torch.log(all_gates.mean(dim=0) + 1e-8)).item()
    return utilization, entropy

if moe_paths:
    print("Collecting BUSI Routing Stats...")
    busi_util, busi_ent = collect_routing_stats(model_moe, busi_loader)
    print("Collecting BUSBRA Routing Stats...")
    busbra_util, busbra_ent = collect_routing_stats(model_moe, busbra_loader)
    
    print("\n=== ROUTING STATISTICS ===")
    print(f"BUSI    -> E0: {busi_util[0]:.1f}%, E1: {busi_util[1]:.1f}%, E2: {busi_util[2]:.1f}%, E3: {busi_util[3]:.1f}%, Entropy: {busi_ent:.4f}")
    print(f"BUSBRA  -> E0: {busbra_util[0]:.1f}%, E1: {busbra_util[1]:.1f}%, E2: {busbra_util[2]:.1f}%, E3: {busbra_util[3]:.1f}%, Entropy: {busbra_ent:.4f}")

In [ ]:
# 5. VISUALIZATION AND QUALITATIVE COMPARISON
os.makedirs("results/phase9/qualitative", exist_ok=True)

def save_comparison(dataset, name, indices):
    fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3*len(indices)))
    if len(indices) == 1: axes = [axes]
    
    for row_idx, data_idx in enumerate(indices):
        img_tensor, mask_tensor = dataset[data_idx]
        
        with torch.no_grad():
            img_batch = img_tensor.unsqueeze(0).to(device)
            v_pred = (model_vanilla(img_batch).argmax(dim=1) > 0).long().squeeze().cpu().numpy() if vanilla_paths else np.zeros((224, 224))
            m_pred = (model_moe(img_batch).argmax(dim=1) > 0).long().squeeze().cpu().numpy() if moe_paths else np.zeros((224, 224))
            
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
        
        axes[row_idx][0].imshow(img_np)
        axes[row_idx][0].set_title(f"{name} {data_idx} - Image")
        axes[row_idx][0].axis("off")
        
        axes[row_idx][1].imshow(mask_tensor.numpy(), cmap="gray")
        axes[row_idx][1].set_title("Ground Truth")
        axes[row_idx][1].axis("off")
        
        axes[row_idx][2].imshow(v_pred, cmap="gray")
        axes[row_idx][2].set_title("Vanilla Swin")
        axes[row_idx][2].axis("off")
        
        axes[row_idx][3].imshow(m_pred, cmap="gray")
        axes[row_idx][3].set_title("Standard-MLP MoE")
        axes[row_idx][3].axis("off")
        
    plt.tight_layout()
    plt.savefig(f"results/phase9/qualitative/{name}_comparison.png")
    plt.close()

# Random sample for qualitative review
np.random.seed(42)
save_comparison(busbra_dataset, "BUSBRA", np.random.choice(len(busbra_dataset), 5, replace=False))
print("Qualitative visualizations saved to results/phase9/qualitative/")